# UGRDarijaEEG — full run on Kaggle

**Three code cells, nothing else.** The whole pipeline lives in `scripts/run_all.py`
inside the repository; this notebook only clones the repository and calls it.

This is deliberate: Kaggle stores notebook cells on Kaggle, not in the repository,
so a notebook that re-implements pipeline steps silently drifts away from the code.
Here there is nothing to keep in sync — cell 2 prints the commit that was cloned.

## One-time setup

1. **Data.** *Add Input* → the dataset containing the eight `.xdf` recordings.
2. **Accelerator: None.** Nothing in the pipeline uses a GPU.
3. **Internet: On** (Settings → Internet), needed to clone the repository.

## Resuming after the 12 h limit

`/kaggle/working` does not survive between two *Save & Run All*:

1. Let the run finish, even if cut — its files remain in **Output**.
2. *Add Input* → **Notebook Output** → this notebook.
3. Run again. Epoch archives, CSVs and predictions already computed are recovered
   and validated, and finished blocks are skipped.

The closing summary **names what is missing** and prints the `--only` command that
recomputes just that.

## One archive, one execution

Every decoding analysis uses the same epoch archive (`--archive brut`, without ocular
removal; the ICA archive only enters the with/without-ICA comparison and the nested
selection). The archive name is written to `results/run_manifest.json`, and results from
a run made with another archive, or from a run older than this rule, are **never reused**:
the epoch archives are recovered, the results are recomputed. The full run takes about
14 h, so it needs two *Save & Run All* (the second with the first's output as input).
At the end, `results/chiffres_manuscrit.csv` lists every number of the manuscript with
its source file, and `results/tableau_configurations.csv` every configuration evaluated.


## 1. Pre-flight check

Checks the known causes of failure before any computation. Runs nothing.

In [ ]:
import glob, socket

problems = []

try:
    socket.create_connection(("github.com", 443), timeout=10).close()
    print("[OK]    github.com reachable")
except OSError:
    problems.append("github.com unreachable: Settings -> Internet -> On "
                    "(requires a phone-verified Kaggle account)")

xdf = glob.glob("/kaggle/input/**/*.xdf", recursive=True)
if xdf:
    print(f"[OK]    {len(xdf)} XDF files found in the inputs")
else:
    problems.append("no .xdf under /kaggle/input: Add Input -> the XDF dataset")

for pattern, what in (("epochs*.npz", "epoch archive(s)"),
                      ("results/*.csv", "result tables"),
                      ("results/*.npz", "predictions and confusion matrices")):
    found = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    if found:
        print(f"[INFO]  {len(found)} {what} recoverable from the inputs")

print()
if problems:
    print("=" * 78)
    print(f"{len(problems)} PROBLEM(S) TO FIX BEFORE CONTINUING")
    print("=" * 78)
    for i, m in enumerate(problems, 1):
        print(f"  {i}. {m}")
    raise SystemExit("pre-flight check failed")
print("Everything is in place.")

## 2. Clone, install dependencies, check the commit

**Read the commit printed at the bottom.** If it is not the one you expect, the run that follows would use an outdated version of the code.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO = "SanaeBelfrouh/UGRDarijaEEG"
DEST = Path("/kaggle/working/UGRDarijaEEG")

if DEST.exists():
    shutil.rmtree(DEST)

subprocess.run(["git", "clone", "--depth", "1",
                f"https://github.com/{REPO}.git", str(DEST)], check=True)

os.chdir(DEST)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

print("=" * 78)
print("CLONED COMMIT — check before running:")
print("=" * 78)
print(subprocess.run(["git", "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, "scripts/run_all.py", "--list"],
                     capture_output=True, text=True).stdout)

## 3. The run

One command. Everything resumes, is time-bounded, and a failing step does not take the following ones down.

To redo only part of it: `--only mots15 grille`, or `--skip archives`.

In [ ]:
import subprocess, sys

code = subprocess.run(
    [sys.executable, "-u", "scripts/run_all.py",
     "--input", "/kaggle/input", "--work", "/kaggle/working",
     "--archive", "brut",
     "--max-hours", "11.0"],
).returncode
print(f"\nrun_all.py finished (exit code {code})")